[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bota-nick/SpectralDevicesTest/blob/main/explore.ipynb)

# Multispectral Camera Explorer — MSDC-2-4-CUS4-005
### Spectral Devices Inc. | frame_000362

| Camera | Band | Central Wavelength | FWHM |
|--------|------|--------------------|------|
| RGB | Green | 540 nm | 60 nm |
| RGB | Blue | 470 nm | 60 nm |
| RGB | Red | 630 nm | 60 nm |
| RGB | Green2 | 540 nm | 60 nm |
| BIO | NIR1 | 735 nm | 25 nm |
| BIO | NIR2 | 800 nm | 25 nm |
| BIO | NIR3 | 865 nm | 25 nm |
| BIO | NIR4 | 930 nm | 25 nm |
| SWIR | — | 1125 nm | — |
| SWIR | — | 1275 nm | — |

In [ ]:
# ── Setup: install dependencies and clone repo if running on Colab ──────────
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q tifffile

    # Clone the repo if the data folder isn't already present
    if not os.path.exists("SpectralDevicesTest"):
        !git clone https://github.com/bota-nick/SpectralDevicesTest.git

    DATA_DIR = "SpectralDevicesTest/data"
else:
    DATA_DIR = "data"

print(f"Running in {'Colab' if IN_COLAB else 'local'} mode")
print(f"Data directory: {DATA_DIR}")

In [ ]:
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Original multi-page TIF: shape (4, H, W), band-first
img = tifffile.imread(f"{DATA_DIR}/frame_000362-RGB.tif")   # (4, 1024, 1024) uint16

BANDS = [
    {"idx": 0, "name": "Green",  "cwl": 540, "color": "green"},
    {"idx": 1, "name": "Blue",   "cwl": 470, "color": "steelblue"},
    {"idx": 2, "name": "Red",    "cwl": 630, "color": "red"},
    {"idx": 3, "name": "Green2", "cwl": 540, "color": "darkgreen"},
]

print(f"Shape : {img.shape}  →  (bands, height, width)")
print(f"Dtype : {img.dtype}")
print(f"Range : {img.min()} – {img.max()}")
print()
for b in BANDS:
    ch = img[b["idx"]]
    print(f"  {b['name']:7s} ({b['cwl']} nm) | min={ch.min():5d}  max={ch.max():5d}  mean={ch.mean():.1f}  std={ch.std():.1f}")

## Individual band images

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle("Individual Bands — frame_000362 RGB Camera", fontsize=13, fontweight="bold")

for ax, b in zip(axes, BANDS):
    ch = img[b["idx"]].astype(float)
    im = ax.imshow(ch, cmap="gray", vmin=img.min(), vmax=img.max())
    ax.set_title(f"{b['name']}\n{b['cwl']} nm", fontsize=11)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="DN")

plt.tight_layout()
plt.show()

## Composite RGB image (Red=630nm, Green=540nm, Blue=470nm)

In [ ]:
def norm_band(arr, lo=2, hi=98):
    p_lo, p_hi = np.percentile(arr, lo), np.percentile(arr, hi)
    return np.clip((arr.astype(float) - p_lo) / (p_hi - p_lo), 0, 1)

R = norm_band(img[2])  # Red   630 nm
G = norm_band(img[0])  # Green 540 nm
B = norm_band(img[1])  # Blue  470 nm
rgb_composite = np.stack([R, G, B], axis=-1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("RGB Composite — frame_000362", fontsize=13, fontweight="bold")

axes[0].imshow(rgb_composite)
axes[0].set_title("True colour (R:630 G:540 B:470 nm)\npercentile-stretched", fontsize=10)
axes[0].axis("off")

rgb_linear = np.stack([
    img[2].astype(float) / 4095,
    img[0].astype(float) / 4095,
    img[1].astype(float) / 4095,
], axis=-1).clip(0, 1)
axes[1].imshow(rgb_linear)
axes[1].set_title("True colour — linear 0–4095 stretch", fontsize=10)
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Reference panel analysis — Black & White targets

In [ ]:
import matplotlib.patches as patches

# ROI definitions [row_start:row_end, col_start:col_end]
_ROIS = {
    "Black": {"rows": (695, 710), "cols": (610, 625), "color": "red"},
    "White": {"rows": (495, 520), "cols": (750, 775), "color": "blue"},
}

# ── 1. rgb_composite with bounding boxes ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(rgb_composite)
for label, roi in _ROIS.items():
    r0, r1 = roi["rows"]
    c0, c1 = roi["cols"]
    rect = patches.Rectangle(
        (c0, r0), c1 - c0, r1 - r0,
        linewidth=2, edgecolor=roi["color"], facecolor="none"
    )
    ax.add_patch(rect)
    ax.text(c0, r0 - 5, label, color=roi["color"], fontsize=10, fontweight="bold")

ax.set_title("RGB Composite — reference panels", fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()

# ── 2. Extract DN values from raw img (band-first) ──────────────────────────
wavelengths = [b["cwl"] for b in BANDS]
band_names  = [b["name"] for b in BANDS]

stats = {}
for label, roi in _ROIS.items():
    r0, r1 = roi["rows"]
    c0, c1 = roi["cols"]
    patch = img[:, r0:r1, c0:c1]          # (4, H, W)
    means   = [patch[i].mean()   for i in range(4)]
    medians = [np.median(patch[i]) for i in range(4)]
    stats[label] = {"means": means, "medians": medians, "patch": patch}
    print(f"\n{label} reference  (rows {r0}:{r1}, cols {c0}:{c1})")
    print(f"  {'Band':8s} {'CWL':>7s}  {'Mean':>8s}  {'Median':>8s}  {'Std':>8s}")
    for i, b in enumerate(BANDS):
        std = patch[i].std()
        print(f"  {b['name']:8s} {b['cwl']:>5d} nm  {means[i]:>8.1f}  {medians[i]:>8.1f}  {std:>8.1f}")

# ── 3. Scatter: mean & median vs wavelength ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Reference Panel — Mean & Median DN per Band", fontsize=13, fontweight="bold")

for ax, stat_key, title in zip(axes, ["means", "medians"], ["Mean DN", "Median DN"]):
    for label, roi in _ROIS.items():
        vals = stats[label][stat_key]
        ax.scatter(wavelengths, vals, color=roi["color"], s=100, zorder=3,
                   label=f"{label} ref")
        ax.plot(wavelengths, vals, color=roi["color"], linewidth=1.5,
                linestyle="--", alpha=0.7)
        for wl, v, name in zip(wavelengths, vals, band_names):
            ax.annotate(name, (wl, v), textcoords="offset points",
                        xytext=(5, 5), fontsize=8, color=roi["color"])
    ax.set_xlabel("Central Wavelength (nm)")
    ax.set_ylabel("DN (0–4095)")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(wavelengths)

plt.tight_layout()
plt.show()

## Pixel value histograms per band

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle("Pixel Value Histograms — RGB Camera Bands", fontsize=13, fontweight="bold")


for ax, b in zip(axes.flat, BANDS):
    ch = img[b["idx"]].ravel()
    ax.hist(ch, bins=256, color=b["color"], alpha=0.8, log=True)
    ax.set_title(f"{b['name']} — {b['cwl']} nm", fontsize=11)
    ax.set_xlabel("DN (0–4095)")
    ax.set_ylabel("Pixel count (log)")

    # Overall mean
    ax.axvline(ch.mean(), color="black", linestyle="--", linewidth=1,
               label=f"mean={ch.mean():.0f}")

    # Black & white reference medians
    for label, roi in _ROIS.items():
        r0, r1 = roi["rows"]
        c0, c1 = roi["cols"]
        med = np.median(img[b["idx"], r0:r1, c0:c1])
        ax.axvline(med, color=roi["color"], linestyle=":", linewidth=1.5,
                   label=f"{label} median={med:.0f}")

    ax.legend(fontsize=8)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(500))

plt.tight_layout()
plt.show()

In [ ]:
# ── Sensor assessment from black & white reference medians ──────────────────
print(f"{'Band':8s} {'CWL':>6s}  {'Black med':>10s}  {'White med':>10s}  "
      f"{'Contrast ratio':>14s}  {'White % of max':>15s}  {'Status':>8s}")
print("-" * 80)

for b in BANDS:
    i = b["idx"]
    r0b, r1b = _ROIS["Black"]["rows"]; c0b, c1b = _ROIS["Black"]["cols"]
    r0w, r1w = _ROIS["White"]["rows"]; c0w, c1w = _ROIS["White"]["cols"]

    blk = np.median(img[i, r0b:r1b, c0b:c1b])
    wht = np.median(img[i, r0w:r1w, c0w:c1w])
    contrast = wht / blk if blk > 0 else float("inf")
    pct_max  = wht / 4095 * 100

    # Simple pass/fail criteria
    saturated   = wht >= 4090
    no_signal   = wht < 200
    poor_contrast = contrast < 5
    status = "SATURATED" if saturated else "WEAK" if no_signal else "LOW CONTRAST" if poor_contrast else "OK"

    print(f"{b['name']:8s} {b['cwl']:>5d}nm  {blk:>10.1f}  {wht:>10.1f}  "
          f"{contrast:>14.1f}x  {pct_max:>14.1f}%  {status:>8s}")

print("""
Assessment criteria used:
  Contrast ratio  = White median / Black median  (higher = better separation)
  White % of max  = White median / 4095          (should be 40–80%; <20% weak, >95% saturated)
  SATURATED       : white panel hits sensor ceiling — no headroom, unusable for radiometry
  WEAK            : white panel < 200 DN — too dark, low SNR
  LOW CONTRAST    : contrast ratio < 5x — black and white panels too similar
  OK              : well-exposed with clear separation between black and white
""")

---
## BIO Camera — reference panel search
`frame_000362-bio.tif` — 4 bands: NIR1 (735 nm), NIR2 (800 nm), NIR3 (865 nm), NIR4 (930 nm)  
Shape: (4, 512, 512) — note: half the RGB spatial resolution, so panel locations will differ

In [ ]:
BIO_BANDS = [
    {"idx": 0, "name": "NIR1", "cwl": 735},
    {"idx": 1, "name": "NIR2", "cwl": 800},
    {"idx": 2, "name": "NIR3", "cwl": 865},
    {"idx": 3, "name": "NIR4", "cwl": 930},
]

bio = tifffile.imread(f"{DATA_DIR}/frame_000362-bio.tif")   # (4, 512, 512) uint16

print(f"Shape : {bio.shape}  →  (bands, height, width)")
print(f"Dtype : {bio.dtype}")
print(f"Range : {bio.min()} – {bio.max()}")
print()
for b in BIO_BANDS:
    ch = bio[b["idx"]]
    print(f"  {b['name']}  ({b['cwl']} nm) | min={ch.min():5d}  max={ch.max():5d}  mean={ch.mean():.1f}  std={ch.std():.1f}")

# Display NIR1 (735 nm) with grid — use percentile stretch to see structure
bio_ch = bio[0].astype(float)
p2, p98 = np.percentile(bio_ch, 2), np.percentile(bio_ch, 98)
bio_disp = np.clip((bio_ch - p2) / (p98 - p2), 0, 1)

fig, ax = plt.subplots(figsize=(20, 20))
ax.imshow(bio_disp, cmap="gray")
ax.set_xticks(range(0, bio.shape[2], 50))
ax.set_yticks(range(0, bio.shape[1], 50))
ax.grid(color="yellow", linewidth=0.5, alpha=0.6)
ax.tick_params(labelsize=7)
ax.set_title("BIO Camera — NIR1 (735 nm), percentile-stretched, grid 50 px", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.patches as patches

_BIO_ROIS = {
    "White": {"rows": (245, 255), "cols": (310, 320), "color": "blue"},
    "Black": {"rows": (342, 352), "cols": (235, 245), "color": "red"},
}

# ── 1. NIR1 image with bounding boxes ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(bio_disp, cmap="gray")
for label, roi in _BIO_ROIS.items():
    r0, r1 = roi["rows"]
    c0, c1 = roi["cols"]
    rect = patches.Rectangle(
        (c0, r0), c1 - c0, r1 - r0,
        linewidth=2, edgecolor=roi["color"], facecolor="none"
    )
    ax.add_patch(rect)
    ax.text(c0, r0 - 4, label, color=roi["color"], fontsize=10, fontweight="bold")

ax.set_xticks(range(0, bio.shape[2], 50))
ax.set_yticks(range(0, bio.shape[1], 50))
ax.grid(color="yellow", linewidth=0.5, alpha=0.5)
ax.tick_params(labelsize=7)
ax.set_title("BIO Camera — NIR1 (735 nm) — reference panels", fontsize=12)
plt.tight_layout()
plt.show()

# ── 2. Extract DN stats per band ─────────────────────────────────────────────
bio_wavelengths = [b["cwl"] for b in BIO_BANDS]
bio_band_names  = [b["name"] for b in BIO_BANDS]

bio_stats = {}
for label, roi in _BIO_ROIS.items():
    r0, r1 = roi["rows"]
    c0, c1 = roi["cols"]
    patch = bio[:, r0:r1, c0:c1]
    means   = [patch[i].mean()    for i in range(4)]
    medians = [np.median(patch[i]) for i in range(4)]
    stds    = [patch[i].std()     for i in range(4)]
    bio_stats[label] = {"means": means, "medians": medians}
    print(f"\n{label} reference  (rows {r0}:{r1}, cols {c0}:{c1})")
    print(f"  {'Band':6s} {'CWL':>7s}  {'Mean':>8s}  {'Median':>8s}  {'Std':>8s}")
    for i, b in enumerate(BIO_BANDS):
        print(f"  {b['name']:6s} {b['cwl']:>5d} nm  {means[i]:>8.1f}  {medians[i]:>8.1f}  {stds[i]:>8.1f}")

# ── 3. Scatter: mean & median vs wavelength ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("BIO — Reference Panel Mean & Median DN per Band", fontsize=13, fontweight="bold")

for ax, stat_key, title in zip(axes, ["means", "medians"], ["Mean DN", "Median DN"]):
    for label, roi in _BIO_ROIS.items():
        vals = bio_stats[label][stat_key]
        ax.scatter(bio_wavelengths, vals, color=roi["color"], s=100, zorder=3,
                   label=f"{label} ref")
        ax.plot(bio_wavelengths, vals, color=roi["color"], linewidth=1.5,
                linestyle="--", alpha=0.7)
        for wl, v, name in zip(bio_wavelengths, vals, bio_band_names):
            ax.annotate(name, (wl, v), textcoords="offset points",
                        xytext=(5, 5), fontsize=8, color=roi["color"])
    ax.set_xlabel("Central Wavelength (nm)")
    ax.set_ylabel("DN (0–4095)")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(bio_wavelengths)

plt.tight_layout()
plt.show()

# ── 4. Histograms with reference medians ──────────────────────────────────────
bio_colors = ["darkblue", "purple", "darkorange", "brown"]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle("Pixel Value Histograms — BIO Camera Bands", fontsize=13, fontweight="bold")

for ax, b, col in zip(axes.flat, BIO_BANDS, bio_colors):
    ch = bio[b["idx"]].ravel()
    ax.hist(ch, bins=256, color=col, alpha=0.8, log=True)
    ax.set_title(f"{b['name']} — {b['cwl']} nm", fontsize=11)
    ax.set_xlabel("DN (0–4095)")
    ax.set_ylabel("Pixel count (log)")
    ax.axvline(ch.mean(), color="black", linestyle="--", linewidth=1,
               label=f"mean={ch.mean():.0f}")
    for label, roi in _BIO_ROIS.items():
        r0, r1 = roi["rows"]
        c0, c1 = roi["cols"]
        med = np.median(bio[b["idx"], r0:r1, c0:c1])
        ax.axvline(med, color=roi["color"], linestyle=":", linewidth=1.5,
                   label=f"{label} median={med:.0f}")
    ax.legend(fontsize=8)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(500))

plt.tight_layout()
plt.show()

# ── 5. Sensor assessment ──────────────────────────────────────────────────────
print(f"\n{'Band':6s} {'CWL':>6s}  {'Black med':>10s}  {'White med':>10s}  "
      f"{'Contrast':>10s}  {'White % max':>12s}  {'Status':>10s}")
print("-" * 75)

for b in BIO_BANDS:
    i = b["idx"]
    r0b, r1b = _BIO_ROIS["Black"]["rows"]; c0b, c1b = _BIO_ROIS["Black"]["cols"]
    r0w, r1w = _BIO_ROIS["White"]["rows"]; c0w, c1w = _BIO_ROIS["White"]["cols"]
    blk = np.median(bio[i, r0b:r1b, c0b:c1b])
    wht = np.median(bio[i, r0w:r1w, c0w:c1w])
    contrast = wht / blk if blk > 0 else float("inf")
    pct_max  = wht / 4095 * 100
    saturated     = wht >= 4090
    no_signal     = wht < 200
    poor_contrast = contrast < 5
    status = "SATURATED" if saturated else "WEAK" if no_signal else "LOW CONTRAST" if poor_contrast else "OK"
    print(f"{b['name']:6s} {b['cwl']:>5d}nm  {blk:>10.1f}  {wht:>10.1f}  "
          f"{contrast:>9.1f}x  {pct_max:>11.1f}%  {status:>10s}")

---
## SWIR 1125 nm Camera
`frame_000362-swir-1125.tif` — single band, shape (1024, 1280), ~15-bit dynamic range

In [ ]:
swir1125 = tifffile.imread(f"{DATA_DIR}/frame_000362-swir-1125.tif")  # (1024, 1280) uint16

print(f"Shape : {swir1125.shape}  →  (height, width)")
print(f"Dtype : {swir1125.dtype}")
print(f"Min   : {swir1125.min()}")
print(f"Max   : {swir1125.max()}")
print(f"Mean  : {swir1125.mean():.1f}")
print(f"Std   : {swir1125.std():.1f}")

# Percentile stretch for display
p2, p98 = np.percentile(swir1125, 2), np.percentile(swir1125, 98)
swir1125_disp = np.clip((swir1125.astype(float) - p2) / (p98 - p2), 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(22, 10))
fig.suptitle("SWIR 1125 nm — frame_000362", fontsize=13, fontweight="bold")

# Percentile stretch
axes[0].imshow(swir1125_disp, cmap="gray")
axes[0].set_xticks(range(0, swir1125.shape[1], 50))
axes[0].set_yticks(range(0, swir1125.shape[0], 50))
axes[0].grid(color="yellow", linewidth=0.5, alpha=0.5)
axes[0].tick_params(labelsize=6)
axes[0].set_title(f"Percentile stretch (p2={p2:.0f}, p98={p98:.0f} DN)", fontsize=10)

# Linear stretch
swir1125_linear = swir1125.astype(float) / swir1125.max()
axes[1].imshow(swir1125_linear, cmap="gray")
axes[1].set_xticks(range(0, swir1125.shape[1], 50))
axes[1].set_yticks(range(0, swir1125.shape[0], 50))
axes[1].grid(color="yellow", linewidth=0.5, alpha=0.5)
axes[1].tick_params(labelsize=6)
axes[1].set_title(f"Linear stretch (0–{swir1125.max()} DN)", fontsize=10)

plt.tight_layout()
plt.show()

# Histogram
fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(swir1125.ravel(), bins=256, color="teal", alpha=0.8, log=True)
ax.axvline(swir1125.mean(), color="black", linestyle="--", linewidth=1,
           label=f"mean={swir1125.mean():.0f}")
ax.axvline(np.median(swir1125), color="orange", linestyle=":", linewidth=1.5,
           label=f"median={np.median(swir1125):.0f}")
ax.set_xlabel("DN")
ax.set_ylabel("Pixel count (log)")
ax.set_title("SWIR 1125 nm — pixel value histogram")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.patches as patches

_SWIR1125_ROIS = {
    "White": {"rows": (480, 490), "cols": (765, 775), "color": "blue"},
    "Black": {"rows": (700, 710), "cols": (600, 610), "color": "red"},
}

# ── 1. Image with bounding boxes ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 16))
ax.imshow(swir1125_disp, cmap="gray")
for label, roi in _SWIR1125_ROIS.items():
    r0, r1 = roi["rows"]
    c0, c1 = roi["cols"]
    rect = patches.Rectangle(
        (c0, r0), c1 - c0, r1 - r0,
        linewidth=2, edgecolor=roi["color"], facecolor="none"
    )
    ax.add_patch(rect)
    ax.text(c0, r0 - 6, label, color=roi["color"], fontsize=10, fontweight="bold")

ax.set_xticks(range(0, swir1125.shape[1], 50))
ax.set_yticks(range(0, swir1125.shape[0], 50))
ax.grid(color="yellow", linewidth=0.5, alpha=0.5)
ax.tick_params(labelsize=6)
ax.set_title("SWIR 1125 nm — reference panels", fontsize=12)
plt.tight_layout()
plt.show()

# ── 2. Stats from raw DN (not display-normalised) ────────────────────────────
print(f"SWIR 1125 nm — reference panel stats  (DN range 0–{swir1125.max()})\n")
print(f"  {'Panel':8s}  {'Min':>7s}  {'Max':>7s}  {'Mean':>8s}  {'Median':>8s}  {'Std':>7s}")
print("  " + "-" * 55)

swir_vals = {}
for label, roi in _SWIR1125_ROIS.items():
    r0, r1 = roi["rows"]
    c0, c1 = roi["cols"]
    patch = swir1125[r0:r1, c0:c1]
    swir_vals[label] = {"median": float(np.median(patch)), "mean": float(patch.mean())}
    print(f"  {label:8s}  {patch.min():>7d}  {patch.max():>7d}  "
          f"{patch.mean():>8.1f}  {np.median(patch):>8.1f}  {patch.std():>7.1f}")

# ── 3. Assessment ────────────────────────────────────────────────────────────
max_dn   = swir1125.max()
blk      = swir_vals["Black"]["median"]
wht      = swir_vals["White"]["median"]
contrast = wht / blk if blk > 0 else float("inf")
pct_max  = wht / max_dn * 100

saturated     = wht >= max_dn * 0.99
no_signal     = wht < 500
poor_contrast = contrast < 5
status = "SATURATED" if saturated else "WEAK" if no_signal else "LOW CONTRAST" if poor_contrast else "OK"

print(f"\n  Contrast ratio   : {contrast:.1f}x  (White / Black median)")
print(f"  White % of max   : {pct_max:.1f}%  (max DN = {max_dn})")
print(f"  Status           : {status}")

# ── 4. Histogram with reference medians ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(swir1125.ravel(), bins=256, color="teal", alpha=0.8, log=True)
ax.axvline(swir1125.mean(), color="black", linestyle="--", linewidth=1,
           label=f"image mean={swir1125.mean():.0f}")
for label, roi in _SWIR1125_ROIS.items():
    r0, r1 = roi["rows"]; c0, c1 = roi["cols"]
    med = np.median(swir1125[r0:r1, c0:c1])
    ax.axvline(med, color=roi["color"], linestyle=":", linewidth=1.5,
               label=f"{label} median={med:.0f}")
ax.set_xlabel("DN")
ax.set_ylabel("Pixel count (log)")
ax.set_title("SWIR 1125 nm — histogram with reference medians")
ax.legend()
plt.tight_layout()
plt.show()

---
## SWIR 1275 nm Camera
`frame_000362-swir-1275.tif` — single band, shape (1024, 1280), same dimensions as SWIR 1125  
Same reference panel locations used: White `[480:490, 765:775]`, Black `[700:710, 600:610]`

In [ ]:
swir1275 = tifffile.imread(f"{DATA_DIR}/frame_000362-swir-1275.tif")  # (1024, 1280) uint16

print(f"Shape : {swir1275.shape}  →  (height, width)")
print(f"Dtype : {swir1275.dtype}")
print(f"Min   : {swir1275.min()}")
print(f"Max   : {swir1275.max()}")
print(f"Mean  : {swir1275.mean():.1f}")
print(f"Std   : {swir1275.std():.1f}")

# Note: image is shifted relative to SWIR 1125 — different ROI coordinates
_SWIR1275_ROIS = {
    "White": {"rows": (480, 490), "cols": (695, 705), "color": "blue"},
    "Black": {"rows": (700, 710), "cols": (520, 530), "color": "red"},
}

# Percentile stretch for display
p2, p98 = np.percentile(swir1275, 2), np.percentile(swir1275, 98)
swir1275_disp = np.clip((swir1275.astype(float) - p2) / (p98 - p2), 0, 1)

# ── 1. Side-by-side images ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(22, 10))
fig.suptitle("SWIR 1275 nm — frame_000362", fontsize=13, fontweight="bold")

axes[0].imshow(swir1275_disp, cmap="gray")
axes[0].set_xticks(range(0, swir1275.shape[1], 50))
axes[0].set_yticks(range(0, swir1275.shape[0], 50))
axes[0].grid(color="yellow", linewidth=0.5, alpha=0.5)
axes[0].tick_params(labelsize=6)
axes[0].set_title(f"Percentile stretch (p2={p2:.0f}, p98={p98:.0f} DN)", fontsize=10)

swir1275_linear = swir1275.astype(float) / swir1275.max()
axes[1].imshow(swir1275_linear, cmap="gray")
axes[1].set_xticks(range(0, swir1275.shape[1], 50))
axes[1].set_yticks(range(0, swir1275.shape[0], 50))
axes[1].grid(color="yellow", linewidth=0.5, alpha=0.5)
axes[1].tick_params(labelsize=6)
axes[1].set_title(f"Linear stretch (0–{swir1275.max()} DN)", fontsize=10)

plt.tight_layout()
plt.show()

# ── 2. Image with reference bounding boxes ───────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 16))
ax.imshow(swir1275_disp, cmap="gray")
for label, roi in _SWIR1275_ROIS.items():
    r0, r1 = roi["rows"]
    c0, c1 = roi["cols"]
    rect = patches.Rectangle(
        (c0, r0), c1 - c0, r1 - r0,
        linewidth=2, edgecolor=roi["color"], facecolor="none"
    )
    ax.add_patch(rect)
    ax.text(c0, r0 - 6, label, color=roi["color"], fontsize=10, fontweight="bold")

ax.set_xticks(range(0, swir1275.shape[1], 50))
ax.set_yticks(range(0, swir1275.shape[0], 50))
ax.grid(color="yellow", linewidth=0.5, alpha=0.5)
ax.tick_params(labelsize=6)
ax.set_title("SWIR 1275 nm — reference panels (shifted from 1125 nm)", fontsize=12)
plt.tight_layout()
plt.show()

# ── 3. Stats from raw DN ─────────────────────────────────────────────────────
print(f"\nSWIR 1275 nm — reference panel stats  (DN range 0–{swir1275.max()})\n")
print(f"  {'Panel':8s}  {'Min':>7s}  {'Max':>7s}  {'Mean':>8s}  {'Median':>8s}  {'Std':>7s}")
print("  " + "-" * 55)

swir1275_vals = {}
for label, roi in _SWIR1275_ROIS.items():
    r0, r1 = roi["rows"]
    c0, c1 = roi["cols"]
    patch = swir1275[r0:r1, c0:c1]
    swir1275_vals[label] = {"median": float(np.median(patch)), "mean": float(patch.mean())}
    print(f"  {label:8s}  {patch.min():>7d}  {patch.max():>7d}  "
          f"{patch.mean():>8.1f}  {np.median(patch):>8.1f}  {patch.std():>7.1f}")

# ── 4. Assessment ────────────────────────────────────────────────────────────
max_dn   = swir1275.max()
blk      = swir1275_vals["Black"]["median"]
wht      = swir1275_vals["White"]["median"]
contrast = wht / blk if blk > 0 else float("inf")
pct_max  = wht / max_dn * 100

saturated     = wht >= max_dn * 0.99
no_signal     = wht < 500
poor_contrast = contrast < 5
status = "SATURATED" if saturated else "WEAK" if no_signal else "LOW CONTRAST" if poor_contrast else "OK"

print(f"\n  Contrast ratio   : {contrast:.1f}x  (White / Black median)")
print(f"  White % of max   : {pct_max:.1f}%  (max DN = {max_dn})")
print(f"  Status           : {status}")

# ── 5. Side-by-side histogram comparison with SWIR 1125 ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
fig.suptitle("SWIR histogram comparison — 1125 nm vs 1275 nm", fontsize=13, fontweight="bold")

for ax, img_sw, name, color, rois, vals in zip(
    axes,
    [swir1125, swir1275],
    ["1125 nm", "1275 nm"],
    ["teal", "darkorchid"],
    [_SWIR1125_ROIS, _SWIR1275_ROIS],
    [swir_vals, swir1275_vals],
):
    ax.hist(img_sw.ravel(), bins=256, color=color, alpha=0.8, log=True)
    ax.axvline(img_sw.mean(), color="black", linestyle="--", linewidth=1,
               label=f"mean={img_sw.mean():.0f}")
    for label, roi in rois.items():
        med = vals[label]["median"]
        ax.axvline(med, color=roi["color"], linestyle=":", linewidth=1.5,
                   label=f"{label} median={med:.0f}")
    ax.set_xlabel("DN")
    ax.set_ylabel("Pixel count (log)")
    ax.set_title(f"SWIR {name}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()